# Matemáticas de la Inteligencia Artificial
## Sesión 8 — Alta dimensión, PCA, SVD y espacios de representación

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/08_pca_embeddings/laboratorio.ipynb)

### Pregunta de la sesión
**¿Cómo descubre una máquina las direcciones relevantes de datos de alta dimensión y cómo las convierte en una representación compacta?**

Recorrido: $\text{alta dimensión}\to\text{centrado}\to\text{covarianza}\to\text{PCA}\to\text{SVD}\to\text{bajo rango}\to\text{embedding}$.

> Completa los bloques `TODO`. Trabajaremos con NumPy y Matplotlib y mantendremos visibles las operaciones matemáticas esenciales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(8)

# 1. Alta dimensión
Para una bola unidad de dimensión $d$, la fracción de volumen en la corona exterior de grosor $\varepsilon$ es $1-(1-\varepsilon)^d$. También estudiaremos cómo las distancias gaussianas se concentran relativamente al crecer $d$.

In [ ]:
def fraccion_corona(d, eps=.1):
    # TODO
    return ...

def estadistica_distancias(d, n_pares=6000, seed=0):
    rg=np.random.default_rng(seed)
    x=rg.normal(size=(n_pares,d)); y=rg.normal(size=(n_pares,d))
    dist=np.linalg.norm(x-y,axis=1)
    # TODO: media, desviación y coeficiente de variación
    media=...; sd=...; cv=...
    return media,sd,cv

dims=[2,5,10,50,100,500]; cvs=[]
for d in dims:
    media,sd,cv=estadistica_distancias(d); cvs.append(cv)
    print(f'd={d:3d} corona={fraccion_corona(d):.6f} media={media:.3f} CV={cv:.4f}')
plt.plot(dims,cvs,marker='o'); plt.xscale('log'); plt.xlabel('d'); plt.ylabel('sd/media'); plt.grid(alpha=.25); plt.show()

# 2. Datos de dimensión 8 con estructura latente de dimensión 2
Construimos un conjunto sintético para comprobar si PCA recupera las direcciones dominantes.

In [ ]:
N=240
t=rng.normal(size=N); s=rng.normal(scale=.6,size=N)
Z_real=np.column_stack([t,s])
A=np.array([[2,.2],[1.7,-.4],[1.4,.7],[1.1,1.2],[-.8,1.5],[-1.3,.9],[.4,-1.6],[1.9,.5]])
desplazamiento=np.array([5.,-3.,2.,7.,0.,1.,-4.,3.])
X=Z_real@A.T+desplazamiento+.16*rng.normal(size=(N,8))
print('X:',X.shape)
plt.scatter(X[:,0],X[:,1],s=18,alpha=.7); plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.grid(alpha=.2); plt.show()

# 3. Centrado, covarianza y PCA
$\mu=N^{-1}\sum_n x_n$, $X_c=X-\mu$ y $C=N^{-1}X_c^TX_c$. Para una dirección unitaria $u$, $u^TCu$ es la varianza proyectada.

In [ ]:
# TODO: centrar y construir C sin np.cov
mu=...
Xc=...
C=...
print('media centrada:',Xc.mean(axis=0))
print('error simetría:',np.max(np.abs(C-C.T)))

# TODO: diagonalizar, ordenar y calcular varianza explicada
valores,vectores=...
orden=...
valores=valores[orden]; vectores=vectores[:,orden]
ratio=...
acumulada=...
k95=...
print('autovalores:',valores)
print('acumulada:',acumulada)
print('k95=',k95)
plt.bar(np.arange(1,9),ratio,alpha=.7); plt.plot(np.arange(1,9),acumulada,marker='o'); plt.axhline(.95,ls='--'); plt.grid(alpha=.2); plt.show()

# 4. Proyección y reconstrucción
Con $V_k$ formado por las primeras componentes: $Z=X_cV_k$ y $\widehat X=ZV_k^T+\mu$.

In [ ]:
k=k95; Vk=vectores[:,:k]
# TODO
Z_pca=...
X_rec=...
rmse=...
print('original',X.shape,'latente',Z_pca.shape,'RMSE',rmse)
plt.scatter(Z_pca[:,0],Z_pca[:,1],s=18,alpha=.7); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(alpha=.2); plt.show()

errores=[]
for k in range(1,9):
    Vk=vectores[:,:k]
    # TODO
    Zk=...; Xk=...; ek=...
    errores.append(ek)
plt.plot(range(1,9),errores,marker='o'); plt.xlabel('k'); plt.ylabel('RMSE'); plt.grid(alpha=.25); plt.show()

# 5. SVD: PCA sin formar la covarianza
Si $X_c=U\Sigma V^T$, entonces $C=V(\Sigma^2/N)V^T$ y $\lambda_i=\sigma_i^2/N$.

In [ ]:
# TODO
U,singulares,Vt=...
lambda_svd=...
print('lambda(C):',valores)
print('sigma^2/N:',lambda_svd)
alineamientos=[abs(vectores[:,i]@Vt[i,:]) for i in range(8)]
print('|v_PCA·v_SVD|:',alineamientos)

# 6. Bajo rango como filtro de ruido
Construimos una señal de rango 2, añadimos ruido y la aproximamos mediante SVD truncada.

In [ ]:
rg=np.random.default_rng(18)
A2=rg.normal(size=(80,2)); B2=rg.normal(size=(60,2))
M_limpia=A2@B2.T
M_ruidosa=M_limpia+.45*rg.normal(size=M_limpia.shape)
# TODO
Um,sm,Vtm=...
M_rango2=...
error_ruido=np.sqrt(np.mean((M_ruidosa-M_limpia)**2))
error_filtrado=np.sqrt(np.mean((M_rango2-M_limpia)**2))
print('RMSE ruidosa:',error_ruido,'RMSE rango-2:',error_filtrado)
plt.semilogy(sm,marker='o'); plt.xlabel('índice'); plt.ylabel('valor singular'); plt.grid(alpha=.25); plt.show()

# 7. Del one-hot al embedding
Una codificación one-hot identifica palabras, pero todas las palabras distintas son ortogonales y están a distancia $\sqrt2$. Construiremos una geometría de baja dimensión mediante una matriz de coocurrencias.

In [ ]:
palabras=['gato','perro','caballo','coche','camión','moto','átomo','electrón','fotón']
contextos=['animal','mascota','granja','carretera','motor','partícula','cuántico','energía']
C_palabras=np.array([[9,8,2,0,0,0,0,1],[8,9,2,0,0,0,0,1],[8,3,9,0,0,0,0,1],[0,0,0,9,8,0,0,3],[0,0,0,8,9,0,0,4],[0,0,0,8,7,0,0,3],[0,0,0,0,0,8,7,4],[0,0,0,0,0,9,8,5],[0,0,0,0,0,8,9,6]],dtype=float)
one_hot=np.eye(len(palabras))
print('d one-hot gato-perro=',np.linalg.norm(one_hot[0]-one_hot[1]))
print('d one-hot gato-fotón=',np.linalg.norm(one_hot[0]-one_hot[-1]))
# TODO: SVD y embedding E=U_3 Sigma_3
Uw,sw,Vtw=...
E=...
plt.scatter(E[:,0],E[:,1],s=60)
for p,(x,y) in zip(palabras,E[:,:2]): plt.text(x+.15,y+.15,p)
plt.grid(alpha=.2); plt.show()

# 8. Similitud coseno
$\operatorname{sim}_{\cos}(a,b)=a^Tb/(\|a\|\|b\|)$. Busca el vecino más próximo de cada palabra.

In [ ]:
def normalizar_filas(X):
    # TODO
    normas=...
    return ...
E_norm=normalizar_filas(E)
S_cos=... # TODO
for i,p in enumerate(palabras):
    fila=S_cos[i].copy(); fila[i]=-np.inf
    j=... # TODO
    print(f'{p:9s} -> {palabras[j]:9s} sim={S_cos[i,j]:.4f}')

# Problema final abierto — Recuperar una representación escondida
Dispones de 360 observaciones de 12 características contaminadas con ruido.

**Parte A.** Centra `X_reto`, calcula su SVD, obtiene la varianza explicada, encuentra el menor $k$ que conserve el 99%, reconstruye para $k=1,\ldots,6$ y compara RMSE frente a `X_reto` y `X_reto_limpia`. ¿Qué $k$ filtra mejor el ruido?

**Parte B.** Construye un embedding de dimensión 3 de `C_reto_palabras`, normalízalo y encuentra vecinos por similitud coseno. Compáralo con one-hot.

**Parte C.** Explica en 6–10 líneas qué significa que una máquina descubra una representación. La solución solo está en el cuaderno docente.

In [ ]:
rg=np.random.default_rng(80); N_reto=360
Z_interna=np.column_stack([rg.normal(scale=1.5,size=N_reto),rg.normal(scale=.9,size=N_reto),rg.normal(scale=.5,size=N_reto)])
B_reto=rg.normal(size=(12,3)); B_reto[:,0]*=1.6; B_reto[:,1]*=1.2; B_reto[:,2]*=.9
mu_reto=np.linspace(-2,3,12)
X_reto_limpia=Z_interna@B_reto.T+mu_reto
X_reto=X_reto_limpia+.12*rg.normal(size=X_reto_limpia.shape)
C_reto_palabras=C_palabras.copy(); palabras_reto=palabras.copy()
print(X_reto.shape,C_reto_palabras.shape)

# TODO — Parte A
mu_r=...; Xr_c=...
Ur,sr,Vtr=...
ratio_r=...; acum_r=...; k99=...
print('k99=',k99)

# TODO — Parte B
...

# Cierre
PCA y SVD nos han dado una primera forma lineal y transparente de aprender representaciones. En las próximas sesiones esas coordenadas se convertirán en parámetros aprendidos y, después, las relaciones entre objetos exigirán grafos y mecanismos de agregación.